In [1]:
 !pip install transformers torch torchaudio
!pip install moviepy

In [2]:
from moviepy.editor import VideoFileClip
import requests
import os
import datetime
from transformers import pipeline
import torch

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



Скачиваем видео через Яндекс диск.

--- В переменную линк добавляем ссылку конкретно на одно видео!



In [3]:
link = 'https://disk.360.yandex.ru/i/R5vceqRnQ-GRjw' #создаем ссылку для каждого видео на просмотр
filename = 'А_03.webm' #называем файл
r = requests.get(f'https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key={link}')
direct = r.json()['href']
os.makedirs('downloads', exist_ok=True)

with open(f'downloads/{filename}', 'wb') as f:
    f.write(requests.get(direct).content)
print(f'Downloaded {filename}')

Downloaded А_03.webm


In [ ]:
def convert_mp4_to_audio(input_path: str, output_path=None) -> None:
    """Конвертирует MP4 видео в MP3 аудио."""
    if output_path is None:
        output_path = input_path.rsplit('.', 1)[0] + '.mp3'

    clip = VideoFileClip(input_path)
    audio = clip.audio

    audio.write_audiofile(output_path, verbose=False, logger=None)

    audio.close()
    clip.close()


Переводим видеофайл в аудио

In [4]:
def convert_webm_to_audio(input_path: str, output_path=None):
    if output_path is None:
        output_path = input_path.rsplit('.', 1)[0] + '.mp3'

    clip = VideoFileClip(input_path)

    audio = clip.audio
    audio.write_audiofile(output_path, verbose=False, logger=None)
    audio.close()

    clip.close()
    return audio

In [6]:
convert_webm_to_audio('/content/downloads/А_03.webm', 'A_03.mp3')

# Обработка видео


1.   Whisper работает,  без тайм кодов и с ними
2.   Vibevoice asr пока не работает через трансформеры,



In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = pipeline("automatic-speech-recognition",  model="openai/whisper-large-v3-turbo", device=device)

The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

In [10]:
audio_path = "/content/A_03.mp3" #аудио
result = pipe(audio_path, return_timestamps=True,
              generate_kwargs={"language": "russian"})

with open("downloads/A_03_transcript.txt", "w") as f: #просто текст
    f.write(result["text"])

[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

In [11]:
def get_pretty_time(timik: tuple) -> str:

  time1, time2 = timik
  new_time = str(datetime.timedelta(seconds= time1)).split('.')[0] + '--' + str(datetime.timedelta(seconds=time2)).split('.')[0]

  return new_time

In [12]:
text_time = ''
for dicts in result["chunks"]:

  new_time = get_pretty_time(dicts['timestamp'])
  text_time += new_time + '\n' + dicts['text'] + '\n'

In [13]:
with open("downloads/A_03_chunks.txt", "w") as f:
    f.write(text_time) #текст с ОЧЕНЬ подробным временем